In [1]:
# combined_delta_iceberg_spark.py
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

# --- your paths & catalog names ---
LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH   = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH   = "/data/data_files/iceberg/iceberg_data_warehouse"
DLT_WAREHOUSE_PATH   = "/data/data_files/iceberg/testdelta"

CATALOG_NAME   = "local"
STG_CATALOG    = "staging"
WH_CATALOG     = "reporting"
DT_CATALOG     = "delta_catalog"

# --- pick the Iceberg runtime that matches your Spark version ---
# Example below is for Spark 3.5 + Scala 2.12 (change if needed)
iceberg_package = "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.3.0"

# If you already installed delta-spark via pip (delta-spark==3.1.0 for Spark 3.5
# or delta-spark==4.0.0 for Spark 4.x), configure_spark_with_delta_pip will help
# wire the Delta extension. We still add the Iceberg runtime via spark.jars.packages.
builder = (
    SparkSession.builder
    .appName("Delta+Iceberg Combined")
    # Put both extensions here (Iceberg first then Delta). Order normally doesn't matter,
    # but both must be present before start.
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
            "io.delta.sql.DeltaSparkSessionExtension")
    # Delta must have its catalog config for spark_catalog
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config(f"spark.sql.catalog.{DT_CATALOG}.warehouse", f"file://{DLT_WAREHOUSE_PATH}")
    
    # Add Iceberg catalogs (HadoopCatalog pointing at local file system)
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file://{LOCAL_WAREHOUSE_PATH}")

    .config(f"spark.sql.catalog.{STG_CATALOG}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{STG_CATALOG}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog")
    .config(f"spark.sql.catalog.{STG_CATALOG}.warehouse", f"file://{STG_WAREHOUSE_PATH}")

    .config(f"spark.sql.catalog.{WH_CATALOG}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{WH_CATALOG}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog")
    .config(f"spark.sql.catalog.{WH_CATALOG}.warehouse", f"file://{RPT_WAREHOUSE_PATH}")

    # Ensure Iceberg jars are on the driver & executors (use correct runtime for your Spark)
    # You can also pass multiple packages comma separated.
    .config("spark.jars.packages", iceberg_package)
)

# Use the helper to configure Delta and then start Spark
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
from pyspark.sql import functions as sf
from pyspark.sql import window as sw
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime, timedelta

In [32]:
# Example: create an Iceberg namespace + table and a Delta path-based table
spark.sql("CREATE NAMESPACE IF NOT EXISTS delta_catalog")

DataFrame[]

In [58]:
# spark.catalog.setCurrentCatalog("Sales")
df_Sales_SalesPerson = spark.table("local.Sales.SalesOrderDetail").alias("sod")

df_Sales_SalesPerson = df_Sales_SalesPerson\
    .withColumn("SalesOrderID", sf.coalesce(sf.col("SalesOrderID").try_cast(sdt.IntegerType()), sf.lit(0)))\
    .withColumn("SalesOrderDetailID", sf.col("SalesOrderDetailID").try_cast(sdt.IntegerType()))\
    .withColumn("CarrierTrackingNumber", sf.col("CarrierTrackingNumber").try_cast(sdt.StringType()))\
    .withColumn("OrderQty", sf.col("OrderQty").try_cast(sdt.IntegerType()))\
    .withColumn("ProductID", sf.col("ProductID").try_cast(sdt.IntegerType()))\
    .withColumn("SpecialOfferID", sf.col("SpecialOfferID").try_cast(sdt.IntegerType()))\
    .withColumn("UnitPrice", sf.coalesce(sf.col("UnitPrice").try_cast(sdt.DecimalType(10, 2)),sf.lit(0).cast(sdt.DecimalType(10, 2))))\
    .withColumn("UnitPriceDiscount", sf.coalesce(sf.col("UnitPriceDiscount").try_cast(sdt.DecimalType(10, 2)),sf.lit(0).cast(sdt.DecimalType(10, 2))))\
    .withColumn("LineTotal", sf.coalesce(sf.col("LineTotal").try_cast(sdt.DecimalType(10, 2)),sf.lit(0).cast(sdt.DecimalType(10, 2))))\
    .withColumn("ModifiedDate", sf.current_date())

df_Sales_SalesPerson.printSchema()

df_Sales_SalesPerson = df_Sales_SalesPerson.drop("rowguid")

df_Sales_SalesPerson.show(5)

root
 |-- SalesOrderID: integer (nullable = false)
 |-- SalesOrderDetailID: integer (nullable = true)
 |-- CarrierTrackingNumber: string (nullable = true)
 |-- OrderQty: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- SpecialOfferID: integer (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- UnitPriceDiscount: decimal(10,2) (nullable = true)
 |-- LineTotal: decimal(10,2) (nullable = true)
 |-- rowguid: string (nullable = true)
 |-- ModifiedDate: date (nullable = false)

+------------+------------------+---------------------+--------+---------+--------------+---------+-----------------+---------+------------+
|SalesOrderID|SalesOrderDetailID|CarrierTrackingNumber|OrderQty|ProductID|SpecialOfferID|UnitPrice|UnitPriceDiscount|LineTotal|ModifiedDate|
+------------+------------------+---------------------+--------+---------+--------------+---------+-----------------+---------+------------+
|           0|                 1|         4911-403C-98|  

In [65]:
# Quick sanity checks
print("Spark version:", spark.version)
spark.sql("SHOW CATALOGS").show(truncate=False)

# Example: create an Iceberg namespace + table and a Delta path-based table
# spark.sql("CREATE NAMESPACE IF NOT EXISTS staging")
# spark.sql("CREATE NAMESPACE IF NOT EXISTS reporting")

# Create Iceberg table in reporting catalog
spark.sql(f"""
          CREATE TABLE delta_catalog.SalesOrderDetail(
          	SalesOrderID int,
			SalesOrderDetailID int,
			CarrierTrackingNumber string,
			OrderQty int,
			ProductID int,
			SpecialOfferID int,
			UnitPrice decimal(10,2),
			UnitPriceDiscount decimal(10,2),
			LineTotal  decimal(10,2),
			ModifiedDate DATE
			) USING delta
          PARTITIONED BY (ModifiedDate)
          LOCATION '{DLT_WAREHOUSE_PATH}/SalesOrderDetail'
		  """)


Spark version: 4.0.1
+---------------+
|catalog        |
+---------------+
|default_iceberg|
|local          |
|spark_catalog  |
+---------------+



DataFrame[]

In [66]:
# Create a small Delta table (path-based) to ensure Delta works
# df_Sales_SalesPerson = spark.table("local.Sales.SalesOrderDetail").alias("sod")
df_Sales_SalesPerson.write.format("delta").mode("append").save(f"{DLT_WAREHOUSE_PATH}/SalesOrderDetail")
print("delta table list:")
spark.sql(f"SHOW TABLES IN {DT_CATALOG}").show(truncate=False)

delta table list:
+-------------+----------------+-----------+
|namespace    |tableName       |isTemporary|
+-------------+----------------+-----------+
|delta_catalog|salesorderdetail|false      |
+-------------+----------------+-----------+



In [70]:
# read Using sql
spark.sql("select SalesOrderID, SalesOrderDetailID, CarrierTrackingNumber, orderqty, productid, specialofferid, unitprice, unitpricediscount, linetotal, modifieddate from delta_catalog.SalesOrderDetail limit 5").show()
# read directly from the path you wrote to
spark.read.format("delta").load(f"{DLT_WAREHOUSE_PATH}/SalesOrderDetail").show(5)
# read using Pyspark
spark.table("delta_catalog.SalesOrderDetail").show(5)

+------------+------------------+---------------------+--------+---------+--------------+---------+-----------------+---------+------------+
|SalesOrderID|SalesOrderDetailID|CarrierTrackingNumber|orderqty|productid|specialofferid|unitprice|unitpricediscount|linetotal|modifieddate|
+------------+------------------+---------------------+--------+---------+--------------+---------+-----------------+---------+------------+
|           0|                 1|         4911-403C-98|       1|      776|             1|  2024.99|             0.00|  2024.99|  2025-12-01|
|           0|                 2|         4911-403C-98|       3|      777|             1|  2024.99|             0.00|  6074.98|  2025-12-01|
|           0|                 3|         4911-403C-98|       1|      778|             1|  2024.99|             0.00|  2024.99|  2025-12-01|
|           0|                 4|         4911-403C-98|       1|      771|             1|  2039.99|             0.00|  2039.99|  2025-12-01|
|           0

In [3]:
# read directly from the path you wrote to
spark.read.format("delta").load(f"{DLT_WAREHOUSE_PATH}/SalesOrderDetail_1").show(5)
print("count:", spark.read.format("delta").load(f"{DLT_WAREHOUSE_PATH}/SalesOrderDetail_1").count())


+------------+------------------+---------------------+--------+---------+--------------+---------+-----------------+---------+------------+
|SalesOrderID|SalesOrderDetailID|CarrierTrackingNumber|OrderQty|ProductID|SpecialOfferID|UnitPrice|UnitPriceDiscount|LineTotal|ModifiedDate|
+------------+------------------+---------------------+--------+---------+--------------+---------+-----------------+---------+------------+
|           0|                 1|         4911-403C-98|       1|      776|             1|  2024.99|             0.00|  2024.99|  2025-12-01|
|           0|                 2|         4911-403C-98|       3|      777|             1|  2024.99|             0.00|  6074.98|  2025-12-01|
|           0|                 3|         4911-403C-98|       1|      778|             1|  2024.99|             0.00|  2024.99|  2025-12-01|
|           0|                 4|         4911-403C-98|       1|      771|             1|  2039.99|             0.00|  2039.99|  2025-12-01|
|           0

In [71]:
# What kind of object is it and where it points
spark.sql("SHOW TABLES IN delta_catalog").show(truncate=False)

# Detailed metadata (look for Location, Provider, Table Type)
spark.sql("DESCRIBE EXTENDED delta_catalog.SalesOrderDetail").show(truncate=False)

# Alternative (gives formatted properties)
spark.sql("DESCRIBE FORMATTED delta_catalog.SalesOrderDetail").show(truncate=False)


+-------------+----------------+-----------+
|namespace    |tableName       |isTemporary|
+-------------+----------------+-----------+
|delta_catalog|salesorderdetail|false      |
+-------------+----------------+-----------+

+----------------------------+--------------------------------------------------------+-------+
|col_name                    |data_type                                               |comment|
+----------------------------+--------------------------------------------------------+-------+
|SalesOrderID                |int                                                     |NULL   |
|SalesOrderDetailID          |int                                                     |NULL   |
|CarrierTrackingNumber       |string                                                  |NULL   |
|OrderQty                    |int                                                     |NULL   |
|ProductID                   |int                                                     |NULL   |
|Speci